In [1]:
#0 Load Libraries and Configurations

import os
import wrds
import pandas as pd
import numpy as np

# ---------- User-configurable paths ----------
PATH_DATA_INTERMEDIATE = "/Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate"  # <-- change this
os.makedirs(PATH_DATA_INTERMEDIATE, exist_ok=True)

OUT_PARQUET = os.path.join(PATH_DATA_INTERMEDIATE, "IBES_Recommendations.parquet")
OUT_CSV     = os.path.join(PATH_DATA_INTERMEDIATE, "IBES_Recommendations.csv")

In [ ]:
#1 Load IBES Data

SQL = """
SELECT
    a.ticker,
    a.estimid,
    a.ereccd,
    a.etext,
    a.ireccd,
    a.itext,
    a.emaskcd,
    a.amaskcd,
    a.anndats,
    a.actdats
FROM ibes.recddet AS a
WHERE a.usfirm = '1'
  AND a.anndats >= DATE '2000-01-01'
;
"""

In [2]:
#2 IBES Data Extraction From WRDS

db = wrds.Connection()
df = db.raw_sql(SQL, date_cols=["anndats", "actdats"])

In [3]:
#3 Data Cleaning

# ---------------- Clean / transform ----------------
# Convert ireccd to numeric (Stata destring ireccd, replace) and drop missing ireccd
df["ireccd"] = pd.to_numeric(df["ireccd"], errors="coerce")
df = df[df["ireccd"].notna()].copy()

# Monthly time index from announcement date
df["time_avail_m"] = df["anndats"].dt.to_period("M").dt.to_timestamp("MS")

# Rename ticker -> tickerIBES
df = df.rename(columns={"ticker": "tickerIBES"})

# Put important columns first (order)
cols_first = ["tickerIBES", "amaskcd", "anndats", "time_avail_m", "ireccd"]
other_cols = [c for c in df.columns if c not in cols_first]
df = df[cols_first + other_cols]

# ---------------- Save ----------------
df.to_parquet(OUT_PARQUET, index=False)
df.to_csv(OUT_CSV, index=False)

print("Saved:")
print(" -", OUT_PARQUET)
print(" -", OUT_CSV)
print(df.head())